In [3]:
# ==============================
# 📌 IMPORT LIBRARIES
# ==============================
import pandas as pd

# ==============================
# 📌 LOAD DATASETS
# ==============================
places_df = pd.read_csv("Top Indian Places to Visit.csv")
cost_df = pd.read_csv("travel cost.csv")

places_df.columns = places_df.columns.str.strip()
cost_df.columns = cost_df.columns.str.strip()


# ==============================
# 📌 HELPER FUNCTIONS
# ==============================

def get_city_attractions(city):
    data = places_df[places_df["City"].str.lower() == city.lower()]

    if data.empty:
        return pd.DataFrame()

    if "Rating" in data.columns:
        data = data.sort_values(by="Rating", ascending=False)

    return data.head(5)


def get_stay_cost(city):
    data = cost_df[cost_df["City"].str.lower() == city.lower()]

    if data.empty:
        return 1500

    col = cost_df.columns[-1]
    # Extract the first number from the cost range string
    cost_str = data.iloc[0][col]
    try:
        # Split by '-' and take the first part, then convert to int
        return int(cost_str.split('-')[0].strip())
    except ValueError:
        return 1500 # Fallback in case parsing fails


# ==============================
# 📌 FOOD COST (FIXED SIMPLE)
# ==============================
def estimate_food_cost():
    return 500  # average daily food


# ==============================
# 📌 SIGHTSEEING COST (SAFE)
# ==============================
def get_sightseeing_cost(city):
    attractions = get_city_attractions(city)

    if attractions.empty:
        return 300

    possible_fee_cols = ["EntryFee", "Entry_Fee", "Fee", "TicketPrice"]

    fee_col = None
    for col in possible_fee_cols:
        if col in attractions.columns:
            fee_col = col
            break

    if fee_col:
        return int(attractions[fee_col].fillna(100).sum())

    return 300


# ==============================
# 📌 SMART BUDGET DISTRIBUTION
# ==============================
def calculate_budget_by_range(city, days, user_budget_per_day):

    # Base costs
    hotel = get_stay_cost(city)
    food = estimate_food_cost()
    sightseeing = get_sightseeing_cost(city)
    transport = 300

    base_total = hotel + food + transport + sightseeing

    # Scale factor to fit user's budget
    scale = user_budget_per_day / base_total

    hotel = int(hotel * scale)
    food = int(food * scale)
    transport = int(transport * scale)
    sightseeing = int(sightseeing * scale)

    per_day = hotel + food + transport + sightseeing
    total = per_day * days

    return {
        "hotel": hotel,
        "food": food,
        "transport": transport,
        "sightseeing": sightseeing,
        "per_day": per_day,
        "total": total
    }


# ==============================
# 📌 ITINERARY (FIXED COLUMN)
# ==============================
def generate_itinerary(city, days):

    attractions = get_city_attractions(city)

    possible_columns = ["Place", "Place_Name", "Destination", "Attraction", "Name"]

    place_col = None
    for col in possible_columns:
        if col in attractions.columns:
            place_col = col
            break

    itinerary = []

    for i in range(days):
        if not attractions.empty and place_col:
            place = attractions.iloc[i % len(attractions)][place_col]
        else:
            place = "Local attractions"

        day_plan = f"""
Day {i+1}:
🌄 Visit: {place}
🍽 Try local food
🌆 Explore markets / relax
        """

        itinerary.append(day_plan.strip())

    return itinerary


# ==============================
# 📌 FINAL AGENT
# ==============================
def travel_agent():

    city = input("Enter city: ")
    days = int(input("Enter number of days: "))
    budget_per_day = int(input("Enter your budget per day (₹): "))

    budget = calculate_budget_by_range(city, days, budget_per_day)
    itinerary = generate_itinerary(city, days)

    print("\n==============================")
    print(f"📍 City: {city}")
    print(f"📅 Days: {days}")
    print(f"💰 Budget per day: ₹{budget_per_day}")
    print("==============================")

    print("\n💵 Budget Breakdown (Per Day):")
    print(f"🏨 Hotel: ₹{budget['hotel']}")
    print(f"🍽 Food: ₹{budget['food']}")
    print(f"🚕 Transport: ₹{budget['transport']}")
    print(f"🎟 Sightseeing: ₹{budget['sightseeing']}")

    print(f"\n💸 Total Cost for {days} days: ₹{budget['total']}")

    print("\n🧳 Your Itinerary:\n")
    for day in itinerary:
        print(day)
        print("-" * 40)


# ==============================
# 📌 RUN
# ==============================
travel_agent()

Enter city: mumbai
Enter number of days: 5
Enter your budget per day (₹): 6000

📍 City: mumbai
📅 Days: 5
💰 Budget per day: ₹6000

💵 Budget Breakdown (Per Day):
🏨 Hotel: ₹4390
🍽 Food: ₹731
🚕 Transport: ₹439
🎟 Sightseeing: ₹439

💸 Total Cost for 5 days: ₹29995

🧳 Your Itinerary:

Day 1:
🌄 Visit: Marine Drive
🍽 Try local food
🌆 Explore markets / relax
----------------------------------------
Day 2:
🌄 Visit: Gateway of India
🍽 Try local food
🌆 Explore markets / relax
----------------------------------------
Day 3:
🌄 Visit: Chhatrapati Shivaji Maharaj Vastu Sangrahalaya
🍽 Try local food
🌆 Explore markets / relax
----------------------------------------
Day 4:
🌄 Visit: Sanjay Gandhi National Park
🍽 Try local food
🌆 Explore markets / relax
----------------------------------------
Day 5:
🌄 Visit: Siddhivinayak Temple
🍽 Try local food
🌆 Explore markets / relax
----------------------------------------
